![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Lab
----

# Going CFC-Free: loading the refrigeration units

## Introduction

The [CFC-Free Initiative](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/sustainability-initiative/overview/#the-cfc-free-initiative) is the first initiative of Coco Pharmaceuticals' sustainability program.  It was suggested by [Zach Now](https://egeria-project.org/practices/coco-pharmaceuticals/personas), founder and leader of the New York Lab, and is led by [Tom Tally](https://egeria-project.org/practices/coco-pharmaceuticals/personas/tom-tally/), the Sustainability Lead.  The aim is to find the refrigeration units across the company that still use chlorofluorocarbon (CFC) refrigerants, which damage the ozone layer and are powerful greenhouse gases, and replace them with more environmentally friendly alternatives.  Most of these units are in the labs and the manufacturing sites.

The refrigeration units are already catalogued in the asset management systems at each site, but none of these systems record the refrigerant installed in each unit.  Tom has collected these details from the sites into a spreadsheet, and plans to use Egeria as the company's asset register for equipment that is relevant to sustainability.  Then governance controls such as [certification types](https://egeria-project.org/types/4/0482-Certifications) and [exception types](https://egeria-project.org/types/4/0455-Exception-Management) can be used to track the progress of the initiative.

There is no open metadata type that can hold the details of a refrigeration unit.  Tom's first proposal was a single new type called *RefrigerationUnit*.  [Erin Overview](https://egeria-project.org/practices/coco-pharmaceuticals/personas/erin-overview/) pointed out that Tom plans to bring in other equipment that is relevant to sustainability, so the team agreed on two new types:

* **Equipment** inherits from [Infrastructure](https://egeria-project.org/types/0/0010-Base-Model) and holds the properties common to all equipment: *make*, *model*, *purchaseDate* and *usedFor*.
* **RefrigerationUnit** inherits from *Equipment* and adds the details needed by the initiative: *refrigerantType*, *refrigerantVolume* and *refrigerantPercentageLossPerYear*.

This notebook follows the work in three steps:

1. [Lemmie Stage](https://egeria-project.org/practices/coco-pharmaceuticals/personas/lemmie-stage/) adds the two new open metadata types to Egeria.
2. Tom sets up a valid metadata list for the *refrigerantType* attribute, so that the refrigerants are recorded consistently.
3. Tom loads the refrigeration units from the spreadsheet, linking each one to the site where it is installed.

----

## Setting up

The first code cell sets up the connection to Egeria.  If you are using the standard quickstart environment then there is no need to change these values.

Lemmie uses the [Metadata Expert](https://egeria-project.org/services/omvs/metadata-expert/overview/) client to add the new types, and the [Valid Metadata](https://egeria-project.org/services/omvs/valid-metadata/overview/) client to check which types already exist.  Tom uses the Valid Metadata client for the valid values, the [Asset Maker](https://egeria-project.org/services/omvs/asset-maker/overview/) client to create the refrigeration units and the [Location Arena](https://egeria-project.org/services/omvs/location-arena/overview/) client to link them to their sites.  The [Classification Explorer](https://egeria-project.org/services/omvs/classification-explorer/overview/) client looks up existing elements by their unique name.

----

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")

lemmies_user_id = "lemmiestage"
toms_user_id = "tomtally"

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for users " + lemmies_user_id + " and " + toms_user_id)
print("\n")

from IPython.display import display, Markdown
import csv
import json

from pyegeria import MetadataExpert, ValidMetadataManager, AssetMaker, LocationArena, ClassificationExplorer
from pyegeria.core._globals import NO_ELEMENTS_FOUND

# Lemmie maintains the open metadata types
lemmies_type_client = MetadataExpert(view_server, url, lemmies_user_id, user_pwd)
lemmies_type_client.create_egeria_bearer_token()

# ... and looks up the types that are already defined
lemmies_valid_metadata_client = ValidMetadataManager(view_server, url, lemmies_user_id, user_pwd)
lemmies_valid_metadata_client.create_egeria_bearer_token()

# Tom maintains the valid values and the asset register
toms_valid_metadata_client = ValidMetadataManager(view_server, url, toms_user_id, user_pwd)
toms_valid_metadata_client.create_egeria_bearer_token()

toms_asset_client = AssetMaker(view_server, url, toms_user_id, user_pwd)
toms_asset_client.create_egeria_bearer_token()

toms_location_client = LocationArena(view_server, url, toms_user_id, user_pwd)
toms_location_client.create_egeria_bearer_token()

# ... and looks up existing elements by their unique name
toms_explorer_client = ClassificationExplorer(view_server, url, toms_user_id, user_pwd)
toms_explorer_client.create_egeria_bearer_token()

print("Connected to Egeria")

----

## Step 1: Adding the new open metadata types

The open metadata types supplied with Egeria are loaded from open metadata archives when the [metadata access store](https://egeria-project.org/concepts/metadata-access-store) starts.  It is also possible to add new types through the API.  These types are validated against the types that are already defined, stored in the metadata access store's local repository and then shared with the other members of its [open metadata repository cohorts](https://egeria-project.org/concepts/cohort-member).

Because the new types are stored in the repository, they only need to be added once.  If this notebook has been run before, the types will already exist, and adding them again would fail.  The function below checks for each type before it creates it.

A new type definition only needs to give the *name* of the types it links to - its supertype and the types of its attributes.  All of the new attributes are strings.

----

In [ ]:
def string_attribute(name: str, description: str) -> dict:
    return {
        "attributeName": name,
        "attributeType": {"class": "OpenMetadataPrimitiveDef", "name": "string"},
        "attributeDescription": description,
    }


def define_entity_type(type_name: str, super_type_name: str, description: str, attributes: list[dict]):
    existing_type = lemmies_valid_metadata_client.get_typedef_by_name(type_name)
    if isinstance(existing_type, dict):
        print(f"Type {type_name} already exists with guid {existing_type.get('guid')}")
        return existing_type.get("guid")

    body = {
        "class": "OpenMetadataEntityDef",
        "name": type_name,
        "description": description,
        "superType": {"name": super_type_name},
        "attributeDefinitions": attributes,
    }
    type_guid = lemmies_type_client.add_type_def(body)
    print(f"Created type {type_name} with guid {type_guid}")
    return type_guid

----

The *Equipment* type is created first because *RefrigerationUnit* names it as its supertype.

----

In [ ]:
equipment_type_guid = define_entity_type(
    "Equipment",
    "Infrastructure",
    "A physical machine or device that is owned by the organization and used in its operations.",
    [
        string_attribute("make", "The manufacturer of the equipment."),
        string_attribute("model", "The manufacturer's model name or number for the equipment."),
        string_attribute("purchaseDate", "The date that the equipment was purchased."),
        string_attribute("usedFor", "A description of how the equipment is used."),
    ])

refrigeration_unit_type_guid = define_entity_type(
    "RefrigerationUnit",
    "Equipment",
    "A piece of equipment that keeps its contents, or the space it serves, cool using a refrigerant.",
    [
        string_attribute("refrigerantType", "The type of refrigerant installed in the unit."),
        string_attribute("refrigerantVolume", "The amount of refrigerant installed in the unit."),
        string_attribute("refrigerantPercentageLossPerYear", "The percentage of the refrigerant that leaks from the unit each year."),
    ])

----

The new *RefrigerationUnit* type can now be retrieved from Egeria in the same way as any other open metadata type.  Requesting the inherited attributes shows that it combines its own attributes with those of *Equipment*, *Infrastructure*, *Asset* and *Referenceable*.

----

In [ ]:
refrigeration_unit_type = lemmies_valid_metadata_client.get_typedef_by_name("RefrigerationUnit", get_inherited_attributes=True)

print("Type: " + refrigeration_unit_type.get("name") + " (supertype: " + refrigeration_unit_type.get("superType", {}).get("name") + ")")
print(refrigeration_unit_type.get("description"))
print()
for attribute in refrigeration_unit_type.get("attributeDefinitions", []):
    print("  " + attribute.get("attributeName"))

----

## Step 2: Setting up the valid values for refrigerantType

Tom's spreadsheet is in the [data](data) folder.  Each row describes one refrigeration unit.  The first part of the unit id identifies the site where the unit is installed.

----

In [ ]:
refrigeration_units_file = "data/coco_refrigeration_units.csv"

with open(refrigeration_units_file, newline="", encoding="utf-8") as csv_file:
    refrigeration_units = list(csv.DictReader(csv_file))

print(f"{len(refrigeration_units)} refrigeration units read from {refrigeration_units_file}")

refrigerant_types = sorted({unit["refrigerant_type"] for unit in refrigeration_units})
print("Refrigerant types in use: " + ", ".join(refrigerant_types))

----

The sites each recorded the refrigerant in their own way.  To make sure every unit uses the same name for the same refrigerant, Tom sets up a [valid metadata value](https://egeria-project.org/guides/planning/valid-values/overview/) for each refrigerant type found in the spreadsheet.  The valid values are attached to the *refrigerantType* attribute of the *RefrigerationUnit* type.

Each valid value also records the family of the refrigerant, its ozone depletion potential (ODP, relative to R-11) and its global warming potential (GWP, relative to carbon dioxide over 100 years).  The refrigerants in the *CFC* family identify the units the initiative must replace.  The GWP shows which of the remaining units to tackle next.

Setting up a valid metadata value creates it if it does not exist, and updates it if it does, so this step can be safely rerun.

----

In [ ]:
refrigerant_details = {
    "R-11":     {"family": "CFC",         "odp": "1.0",  "gwp": "4750",  "description": "Trichlorofluoromethane.  A CFC used in older low-pressure centrifugal chillers.  Its production was banned under the Montreal Protocol."},
    "R-12":     {"family": "CFC",         "odp": "1.0",  "gwp": "10900", "description": "Dichlorodifluoromethane.  A CFC used in older fridges and freezers.  Its production was banned under the Montreal Protocol."},
    "R-500":    {"family": "CFC",         "odp": "0.74", "gwp": "8077",  "description": "A CFC blend of R-12 and R-152a used in older centrifugal chillers.  Its production was banned under the Montreal Protocol."},
    "R-502":    {"family": "CFC",         "odp": "0.33", "gwp": "4657",  "description": "A CFC blend of R-22 and R-115 used in older low-temperature freezers.  Its production was banned under the Montreal Protocol."},
    "R-134a":   {"family": "HFC",         "odp": "0",    "gwp": "1430", "description": "Tetrafluoroethane.  A common hydrofluorocarbon (HFC) found in older lab fridges and large chillers."},
    "R-1234ze": {"family": "HFO",         "odp": "0",    "gwp": "7",    "description": "Tetrafluoropropene.  A low-GWP hydrofluoroolefin (HFO) used as a replacement for R-134a in chillers."},
    "R-170":    {"family": "Hydrocarbon", "odp": "0",    "gwp": "6",    "description": "Ethane.  A natural refrigerant used in ultra-low temperature freezers."},
    "R-290":    {"family": "Hydrocarbon", "odp": "0",    "gwp": "3",    "description": "Propane.  A natural refrigerant used in -80°C freezers."},
    "R-32":     {"family": "HFC",         "odp": "0",    "gwp": "675",  "description": "Difluoromethane.  An HFC with a lower GWP than R-410A, used in process chillers and VRV systems."},
    "R-404A":   {"family": "HFC blend",   "odp": "0",    "gwp": "3922", "description": "A high-GWP HFC blend used in older walk-in cold rooms.  It is being phased out under the F-Gas regulations."},
    "R-410A":   {"family": "HFC blend",   "odp": "0",    "gwp": "2088", "description": "An HFC blend used in rooftop HVAC units."},
    "R-600a":   {"family": "Hydrocarbon", "odp": "0",    "gwp": "3",    "description": "Isobutane.  A natural refrigerant used in pharmacy-grade fridges."},
}

for refrigerant_type in refrigerant_types:
    details = refrigerant_details.get(refrigerant_type, {"description": "Refrigerant type " + refrigerant_type + "."})
    body = {
        "displayName": refrigerant_type,
        "description": details.get("description"),
        "preferredValue": refrigerant_type,
        "dataType": "string",
        "isCaseSensitive": False,
        "isDeprecated": False,
    }
    if "family" in details:
        body["additionalProperties"] = {"refrigerantFamily": details["family"],
                                        "ozoneDepletionPotential": details["odp"],
                                        "globalWarmingPotential": details["gwp"]}

    toms_valid_metadata_client.setup_valid_metadata_value("refrigerantType", "RefrigerationUnit", body)
    print("Set up valid value " + refrigerant_type + " for RefrigerationUnit.refrigerantType")

----

The valid values can be retrieved from Egeria, for example, to populate a drop-down list in a user interface.

----

In [ ]:
valid_values = toms_valid_metadata_client.get_valid_metadata_values("refrigerantType", "RefrigerationUnit")

if isinstance(valid_values, list):
    for valid_value in valid_values:
        additional_properties = valid_value.get("additionalProperties", {})
        print(f"{valid_value.get('preferredValue'):10} {additional_properties.get('refrigerantFamily', ''):12} "
              f"ODP {additional_properties.get('ozoneDepletionPotential', '?'):>4}  GWP {additional_properties.get('globalWarmingPotential', '?'):>5}  "
              f"{valid_value.get('description')}")
else:
    print(valid_values)

----

## Step 3: Loading the refrigeration units

Each refrigeration unit becomes a *RefrigerationUnit* asset.  The unique name (qualifiedName) of each unit is built from its unit id.  The properties are passed in the *InfrastructureProperties* class, which is the closest class to *RefrigerationUnit* that pyegeria and Egeria know about.  It includes the *deploymentStatus* attribute inherited from *Infrastructure*, which is set from the spreadsheet.  All of the units are currently in use, so they are *ACTIVE*.  The attributes introduced by the new types are not known to this class, so they are passed in the `extendedProperties` map, which maps the attribute name to the attribute value.

Each unit is its own [anchor](https://egeria-project.org/concepts/anchor) because it is managed independently of any other element.  It is linked to the site where it is installed using the [KnownLocation](https://egeria-project.org/types/0/0025-Locations) relationship.  The site is identified by the middle part of the unit id.  These sites were set up by the earlier workbooks.

----

In [ ]:
site_qualified_names = {
    "NYC": "Site::NY-SITE",
    "AUS": "Site::AUS-SITE",
    "EDM": "Site::ED-SITE",
    "WIN": "Site::WINCH-SITE",
    "LON": "Site::LDN-SITE",
    "AMS": "Site::AMS-SITE",
    "BUC": "Site::BUC-SITE",
}

site_guids = {}
for site_code, site_qualified_name in site_qualified_names.items():
    site_guid = toms_explorer_client.get_element_guid_by_unique_name(site_qualified_name)
    if site_guid is None or site_guid == NO_ELEMENTS_FOUND:
        print(f"Site {site_qualified_name} not found - units for {site_code} will not be linked to their site")
    else:
        site_guids[site_code] = site_guid
        print(f"{site_code}: {site_qualified_name} is {site_guid}")

----

The units are now created.  As with the types, the code checks whether each unit already exists so that this notebook can be rerun:

* A unit that already exists is updated with the values in the spreadsheet.  The update is a *merge update*, so only the properties supplied are changed.
* A unit that has been archived - for example, the CFC unit replaced by the New York team in [ny-refrigeration-unit-replacement.ipynb](ny-refrigeration-unit-replacement.ipynb) - is left alone.  Archived elements are not returned by normal queries, and Egeria renames an element when it archives it, adding `_archivedOn_` and the date to its unique name.  So the code searches for a name that starts with the unit's original name followed by `_archivedOn_`, setting `forLineage` to include archived elements in the search.

----

In [ ]:
def get_unit_guid(qualified_name: str) -> str | None:
    unit_guid = toms_explorer_client.get_element_guid_by_unique_name(qualified_name)
    if unit_guid is None or unit_guid == NO_ELEMENTS_FOUND:
        return None
    return unit_guid


def get_archived_unit_guid(qualified_name: str) -> str | None:
    body = {
        "class": "SearchStringRequestBody",
        "searchString": qualified_name + "_archivedOn_",
        "startsWith": True,
        "ignoreCase": False,
        "metadataElementTypeName": "RefrigerationUnit",
        "forLineage": True,
    }
    archived_units = toms_asset_client.find_assets(body=body, output_format="JSON")
    if isinstance(archived_units, list) and archived_units:
        return archived_units[0]["elementHeader"]["guid"]
    return None


def unit_properties(unit: dict) -> dict:
    return {
        "class": "InfrastructureProperties",
        "typeName": "RefrigerationUnit",
        "qualifiedName": "RefrigerationUnit::" + unit["unit_id"],
        "displayName": unit["unit_id"],
        "description": unit["make"] + " " + unit["model"] + ": " + unit["used_for"],
        "deploymentStatus": unit["deployment_status"],
        "extendedProperties": {
            "make": unit["make"],
            "model": unit["model"],
            "purchaseDate": unit["purchase_date"],
            "usedFor": unit["used_for"],
            "refrigerantType": unit["refrigerant_type"],
            "refrigerantVolume": unit["refrigerant_volume"],
            "refrigerantPercentageLossPerYear": unit["refrigerant_percentage_loss_per_year"],
        },
    }


unit_guids = {}
for unit in refrigeration_units:
    unit_id = unit["unit_id"]
    qualified_name = "RefrigerationUnit::" + unit_id
    site_code = unit_id.split("-")[1]

    unit_guid = get_unit_guid(qualified_name)
    if unit_guid:
        body = {
            "class": "UpdateElementRequestBody",
            "mergeUpdate": True,
            "properties": unit_properties(unit),
        }
        toms_asset_client.update_asset(unit_guid, body)
        print(f"{unit_id} already exists with guid {unit_guid} - updated from the spreadsheet")
    elif get_archived_unit_guid(qualified_name):
        print(f"{unit_id} has been archived - skipped")
        continue
    else:
        body = {
            "class": "NewElementRequestBody",
            "isOwnAnchor": True,
            "properties": unit_properties(unit),
        }
        unit_guid = toms_asset_client.create_asset(asset_type=["InfrastructureProperties"], body=body)

        if site_code in site_guids:
            toms_location_client.link_known_location(unit_guid, site_guids[site_code])
            print(f"{unit_id} created with guid {unit_guid} and linked to {site_qualified_names[site_code]}")
        else:
            print(f"{unit_id} created with guid {unit_guid} - no site link")

    unit_guids[unit_id] = unit_guid

print(f"\n{len(unit_guids)} active refrigeration units are in Egeria")

----

## Reviewing the results

Retrieving one of the units shows that the new attributes are returned with the standard asset properties, along with the site the unit is linked to.

----

In [ ]:
unit = toms_asset_client.get_asset_by_guid(unit_guids["REF-NYC-005"], output_format="JSON")

print(json.dumps(unit.get("properties"), indent=2, ensure_ascii=False))

for known_location in unit.get("knownLocations", []):
    print("\nLocated at: " + known_location.get("relatedElement", {}).get("properties", {}).get("displayName", ""))

----

Tom can now see which units still use CFCs at each site.  These are the units the initiative needs to replace.  The units using other high-GWP refrigerants are candidates for a later phase.

----

In [ ]:
high_gwp_threshold = 1000

def refrigerant_family(unit: dict) -> str:
    return refrigerant_details.get(unit["refrigerant_type"], {}).get("family", "")


def refrigerant_gwp(unit: dict) -> int:
    return int(refrigerant_details.get(unit["refrigerant_type"], {}).get("gwp", "0"))


report = "| Site | Units | CFC units | Other high-GWP units |\n|---|---|---|---|\n"
for site_code, site_qualified_name in site_qualified_names.items():
    site_units = [u for u in refrigeration_units if u["unit_id"].split("-")[1] == site_code and u["unit_id"] in unit_guids]
    cfc_units = [u["unit_id"] + " (" + u["refrigerant_type"] + ")" for u in site_units if refrigerant_family(u) == "CFC"]
    high_gwp_units = [u["unit_id"] + " (" + u["refrigerant_type"] + ")" for u in site_units
                      if refrigerant_family(u) != "CFC" and refrigerant_gwp(u) >= high_gwp_threshold]
    report += f"| {site_qualified_name} | {len(site_units)} | {', '.join(cfc_units)} | {', '.join(high_gwp_units)} |\n"

display(Markdown(report))

----

## Next steps

The refrigeration units are now in Egeria's asset register, each linked to its site, so the units can be browsed by site in [egeria-explorer](https://egeria-project.org/user-interfaces/egeria-explorer/overview).  The New York team is the first to replace its CFC unit - see [ny-refrigeration-unit-replacement.ipynb](ny-refrigeration-unit-replacement.ipynb).

----
License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/),
Copyright Contributors to the ODPi Egeria project.